In [0]:
%sql
CREATE OR REPLACE FUNCTION dbr_dev.skybound_gold.rls_country_filter(origin_country STRING)
  RETURNS BOOLEAN
  RETURN 
  IF(
    current_user() IN (
      'yuriy.verbitskyi@softserve.academy', 'valeriialiebiedie179@softserve.academy'
    ),
    TRUE,
    origin_country IN (
      'United Kingdom',
      'Germany',
      'France',
      'Spain',
      'Italy',
      'Ireland',
      'Austria',
      'Kingdom of the Netherlands',
      'Poland',
      'Switzerland',
      'Sweden',
      'Portugal',
      'Norway',
      'Belgium',
      'Greece',
      'Hungary',
      'Czech Republic',
      'Denmark',
      'Finland',
      'Luxembourg',
      'Latvia',
      'Bulgaria',
      'Iceland',
      'Romania',
      'Serbia',
      'Estonia',
      'Lithuania',
      'Croatia',
      'Slovakia',
      'Republic of Moldova',
      'Ukraine',
      'Slovenia',
      'Montenegro',
      'Bosnia and Herzegovina',
      'Monaco',
      'San Marino',
      'Malta',
      'Cyprus',
      'Turkey'
    )
  )

In [0]:
%sql
ALTER MATERIALIZED VIEW dbr_dev.skybound_gold.gold_active_flights SET
ROW FILTER dbr_dev.skybound_gold.rls_country_filter ON (origin_country)

In [0]:
%sql
SELECT
  current_user() AS who_am_i,
  count(*) AS total_flights,
  count(DISTINCT origin_country) AS countries
FROM
  dbr_dev.skybound_gold.gold_active_flights

In [0]:
%sql
CREATE OR REPLACE FUNCTION dbr_dev.skybound_gold.mask_icao24(icao24 STRING)
RETURNS STRING
RETURN
  CASE
    WHEN current_user() IN (
      'yuriy.verbitskyi@softserve.academy',
      'valeriialiebiedie179@softserve.academy'
    ) THEN icao24
    WHEN current_user() = 'k.ilya.v@softserve.academy'
      THEN CONCAT(LEFT(icao24, 4), '**')
    ELSE '******'
  END;

CREATE OR REPLACE FUNCTION dbr_dev.skybound_gold.mask_callsign(callsign STRING)
RETURNS STRING
RETURN
  CASE
    WHEN current_user() IN (
      'yuriy.verbitskyi@softserve.academy',
      'valeriialiebiedie179@softserve.academy'
    ) THEN callsign
    WHEN current_user() = 'k.ilya.v@softserve.academy'
      THEN CONCAT(LEFT(callsign, 3), '***')
    ELSE '******'
  END;

In [0]:
%sql
ALTER MATERIALIZED VIEW dbr_dev.skybound_gold.gold_active_flights
  ALTER COLUMN icao24 SET MASK dbr_dev.skybound_gold.mask_icao24;

ALTER MATERIALIZED VIEW dbr_dev.skybound_gold.gold_active_flights
  ALTER COLUMN callsign SET MASK dbr_dev.skybound_gold.mask_callsign;

## Summary: What each user sees

| Column | Yuriy (admin) | Valeriia (admin) | Ilya (viewer) | Others |
|--------|--------------|-----------------|--------------|--------|
| **Rows** | All 8,212 (94 countries) | All 8,212 | ~7,026 (Europe only) | ~7,026 (Europe only) |
| `icao24` | `3c6752` | `3c6752` | `3c67**` | `******` |
| `callsign` | `MSR727` | `MSR727` | `MSR***` | `******` |
| `registration` | `SU-GDU` | `SU-GDU` | **NULL** | **NULL** |
| `operator` | `Egyptair` | `Egyptair` | **NULL** | **NULL** |
| `latitude` | exact | exact | exact | exact |
| `longitude` | exact | exact | exact | exact |

In [0]:
%sql
-- ROLLBACK:

-- ALTER MATERIALIZED VIEW dbr_dev.skybound_gold.gold_active_flights DROP ROW FILTER;
-- ALTER MATERIALIZED VIEW dbr_dev.skybound_gold.gold_active_flights ALTER COLUMN registration DROP MASK;
-- ALTER MATERIALIZED VIEW dbr_dev.skybound_gold.gold_active_flights ALTER COLUMN operator DROP MASK;
-- ALTER MATERIALIZED VIEW dbr_dev.skybound_gold.gold_active_flights ALTER COLUMN icao24 DROP MASK;
-- ALTER MATERIALIZED VIEW dbr_dev.skybound_gold.gold_active_flights ALTER COLUMN callsign DROP MASK;

-- DROP FUNCTION IF EXISTS dbr_dev.skybound_gold.rls_country_filter;
-- DROP FUNCTION IF EXISTS dbr_dev.skybound_gold.cls_mask_registration;
-- DROP FUNCTION IF EXISTS dbr_dev.skybound_gold.cls_mask_operator;
-- DROP FUNCTION IF EXISTS dbr_dev.skybound_gold.mask_icao24;
-- DROP FUNCTION IF EXISTS dbr_dev.skybound_gold.mask_callsign;